In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os, json

import sys
sys.path.append('../') 

from main import (
    section_splitter,
    indexer,
    run_conflict_detection,
    argument_map,
    export_map,
    save_session,
    load_session,
    create_session_id,
    steelman,
    challenge,
    answer_for
)

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
# ──  Upload papers, split sections and index ───
PDF_A = "./paper1.pdf"  # ← change these
PDF_B = "./paper2.pdf"

sections_a = section_splitter(PDF_A)
sections_b = section_splitter(PDF_B)

index_a=indexer(sections_a,128)
index_b=indexer(sections_b,128)

print("Paper A sections:", {k: f"pp.{v['pages'][0]}-{v['pages'][1]}" for k,v in sections_a.items()})
print("Paper B sections:", {k: f"pp.{v['pages'][0]}-{v['pages'][1]}" for k,v in sections_b.items()})

Paper A sections: {'info': 'pp.1-1', 'abs': 'pp.1-1', 'body': 'pp.1-4', 'conclusion': 'pp.4-4'}
Paper B sections: {'info': 'pp.1-1', 'abs': 'pp.1-1', 'body': 'pp.1-4', 'conclusion': 'pp.4-6'}


In [6]:
# ── Conflict detection ──
sections_a_simp = {'abs': sections_a['abs'], 'conc': sections_a['conclusion']}
sections_b_simp = {'abs': sections_b['abs'], 'conc': sections_b['conclusion']}

conflicts = run_conflict_detection(sections_a, sections_a_simp, sections_b, sections_b_simp)

print(f"{len(conflicts)} conflicts detected:\n")
for i, c in enumerate(conflicts):
    print(f"[{i}] {c['type'].upper()} — {c['concept']}")
    print(f"    {c['summary']}\n")

3 conflicts detected:

[0] METHODOLOGICAL — Evolving dark energy
    Paper A critiques the methods of combining datasets, pointing out tensions, while Paper B uses similar combinations without addressing such tensions.

[1] EMPIRICAL — Tensions Among CMB, BAO, and SN Datasets
    Paper A expresses skepticism about the validity of combining datasets due to tensions, while Paper B supports the use of combined datasets to indicate evolving dark energy, citing constraints that deviate significantly from the ΛCDM model.

[2] EMPIRICAL — Inconsistent Dataset Combinations
    Paper A highlights tensions and inconsistencies between datasets like CMB, DESI DR2, and SN while Paper B suggests that these inconsistencies might be statistical artifacts rather than true anomalies, pointing to different conclusions on dark energy dynamics.



In [7]:
# ──  Build argument map ───
G = argument_map(conflicts, "paper_a", "paper_b")

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

Nodes: 8
Edges: 9


In [8]:
# ── Cell 8: Cross-examination ────────────────────────────
# Get node IDs for first conflict
conflict_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('type') == 'conflict']
node_a_id, node_b_id = conflict_edges[0]

print(f"Selected conflict: {node_a_id} → {node_b_id}\n")

# Pick a tool: steelman | challenge | answer_for
# Pick a paper: "paper_a" | "paper_b"

result = steelman(G, node_a_id, node_b_id, "paper_a", index_a, index_b, client)
print(f"STEELMAN — {result['steelmaned_paper']}")
print(result['response']['Argument'])
print("\nSources:")
for s in result['sources']:
    print(f"  {s['paper_id']} — {s['section']}, pp.{s['page_start']}-{s['page_end']} ({s['score']:.2f})")

Selected conflict: paper_a_conflict_2_evolving_dark_energy → paper_b_conflict_2_evolving_dark_energy

STEELMAN — paper_a
Paper A's position is strengthened by its meticulous analysis, which reveals significant tensions and inconsistencies among the combined datasets used by the DESI collaboration, casting doubt on their conclusions about Dynamical Dark Energy (DDE). By advocating for a more robust approach of analyzing each dataset independently, paper A identifies strong statistical preferences for DDE over the ΛCDM model, suggesting the presence of DDE without the problematic constraints seen in combined datasets. Furthermore, paper A effectively challenges the reliability of claimed evidence for DDE, emphasizing the necessity for future high-precision observations to clarify the role of dark energy in cosmic evolution.

Sources:
  paper_a — abs, pp.1-1 (0.95)
  paper_a — abs, pp.1-1 (0.90)
  paper_a — body, pp.1-4 (0.88)
  paper_a — body, pp.1-4 (0.87)
  paper_a — conclusion, pp.4-4